In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

import shap

# Map x1-x23 to the official UCI variable names so the notebook is self-explanatory.
COLUMN_MAP = {
    "x1": "limit_bal",
    "x2": "sex",
    "x3": "education",
    "x4": "marriage",
    "x5": "age",
    "x6": "pay_0", 
    "x7": "pay_2", 
    "x8": "pay_3", 
    "x9": "pay_4",
    "x10": "pay_5", 
    "x11": "pay_6",
    "x12": "bill_amt1", 
    "x13": "bill_amt2", 
    "x14": "bill_amt3",
    "x15": "bill_amt4", 
    "x16": "bill_amt5", 
    "x17": "bill_amt6",
    "x18": "pay_amt1", 
    "x19": "pay_amt2", 
    "x20": "pay_amt3",
    "x21": "pay_amt4", 
    "x22": "pay_amt5", 
    "x23": "pay_amt6",
}

# fetch the dataset directly from OpenML (cached locally by sklearn after the first run)
raw = fetch_openml(data_id=42477, as_frame=True, parser="auto")
X = raw.data.rename(columns=COLUMN_MAP)
y = raw.target.astype(int)

CLASS_NAMES = ["No Default", "Default"]

# same split as Task 1/2: 80/20, stratified, fixed seed for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [6]:
models = {}

#Model 1: Logistic Regression
models["logistic_regression"] = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced").fit(X_train_scaled, y_train)

#Model 2: Random Forest
models["random_forest"] = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1).fit(X_train_scaled, y_train)

#Model 3: Gradient Boosting
gradient_boosting_sample_weight = compute_sample_weight("balanced", y_train)
models["gradient_boosting"] = GradientBoostingClassifier(random_state=42).fit(X_train_scaled, y_train, sample_weight=gradient_boosting_sample_weight)

performance_rows = []

for name, model in models.items():
    y_prediction = model.predict(X_test_scaled)
    y_probability = model.predict_proba(X_test_scaled)[:, 1]
    performance_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_prediction),
        "auc": roc_auc_score(y_test, y_probability),
    })

performance_df = pd.DataFrame(performance_rows)
performance_df

,model,accuracy,auc
0,logistic_regression,0.679667,0.708115
1,random_forest,0.800833,0.761115
2,gradient_boosting,0.764833,0.779242


In [9]:
N_SAMPLES = 30
TOP_K = 10
LIME_REF_SEED = 42
LIME_SEEDS = [0, 1, 2, 3, 4, 42, 99]

# Same random_state as Task 2 -> same 30 clients
sample_df = X_test_scaled.sample(N_SAMPLES, random_state=42)
sample_indices = sample_df.index.tolist()
print(f"Selected {len(sample_indices)} test clients \n")


def positive_class_shap_values(explainer, X):
    explanation = explainer(X)
    values = explanation.values
    if values.ndim == 3:
        return values[:, :, 1]
    return values

shap_explainers = {
    "logistic_regression": shap.LinearExplainer(models["logistic_regression"], X_train_scaled),
    "random_forest": shap.TreeExplainer(models["random_forest"]),
    "gradient_boosting": shap.TreeExplainer(models["gradient_boosting"]),
}

shap_values_by_model = {}
for name, explainer in shap_explainers.items():
    values = positive_class_shap_values(explainer, sample_df)
    shap_values_by_model[name] = pd.DataFrame(values, columns=X.columns, index=sample_indices)

# Sanity check: top feature for the first client, per model
for name, values_df in shap_values_by_model.items():
    top_feature = values_df.iloc[0].abs().idxmax()
    print(f"{name}: top feature for first client = {top_feature}")

Background dataset has 24000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=24000 when initializing the masker.


Selected 30 test clients 

logistic_regression: top feature for first client = bill_amt1
random_forest: top feature for first client = pay_4
gradient_boosting: top feature for first client = pay_0
